# 05 · cNMF preparation, K selection, and consensus

This notebook prepares exact lineage-specific count matrices, runs an exploratory K sweep, and launches a higher-replicate consensus fit for the selected K.

Choose the cell universe and run controls first. Each execution step is opt-in, so the notebook can be reviewed safely before starting computational work. For unattended runs, use `scripts/run_cnmf_workflow.py` with the same configuration.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from spatial_workflow.cnmf import (
    cnmf_stage_commands,
    cnmf_status,
    export_lineage_counts,
    load_cnmf_context,
    load_k_selection_stats,
    plot_k_selection,
    preview_lineage_selection,
    run_selected,
    run_sweep,
    with_selected_k,
    write_cnmf_tmux_scripts,
)

CONFIG_PATH = REPO_ROOT / "configs" / "local.yaml"

## Analysis and run controls

Select the lineage, annotation column, exact cell-type labels, spatial domains, and conditions to include. Use a distinct `ANALYSIS_NAME` whenever the selected cell universe changes so results from different analyses cannot be mixed.

The run switches default to read-only behavior. Enable only the stage you intend to launch.

In [44]:
LINEAGE = "perivascular"  # astrocyte or oligodendrocyte
ANALYSIS_NAME = None   # required when changing the cell universe
CELL_TYPE_KEY = None   # None uses YAML; e.g. "cluster_sub"
CELL_TYPES = None     # e.g. ["AST-CX", "AST-TH"]
COMPARTMENTS = "all"  # e.g. ["0", "3", "7"]
CONDITIONS = "all"    # or exact condition names
SELECTED_K = 26      # None uses the configured lineage K
WORKERS = None         # runtime override for both collections

RUN_EXPORT = False
RUN_SWEEP = False
RUN_SELECTED_CONSENSUS = True
RUN_IN_TMUX = True    # generate scripts instead of blocking Jupyter
OVERWRITE_EXPORT = False

config, spec, paths = load_cnmf_context(
    CONFIG_PATH,
    LINEAGE,
    cell_type_key=CELL_TYPE_KEY,
    analysis_name=ANALYSIS_NAME,
    cell_types=CELL_TYPES,
    compartments=COMPARTMENTS,
    conditions=CONDITIONS,
    selected_k=SELECTED_K,
    workers=WORKERS,
)

## Confirm the selected cells

This audit reports the exact labels, samples, conditions, and spatial domains that will enter cNMF. Review it before exporting counts or launching a job.

In [45]:
selection = preview_lineage_selection(config, spec, paths.source_h5ad)
for table_name, table in selection.items():
    print(table_name)
    display(table)

overview


,selection
lineage,perivascular
cells,126014
genes_before_filter,5006
cell_type_key,cluster_sub
cell_types,"EC, PVF, PERICYTE"
compartments,all
conditions,all
samples,12


by_sample


,condition,sample_id,n_cells
0,naive_igg,naive_70_igg,10415
1,naive_igg,naive_72_igg,9442
2,naive_igg,naive_75_igg,11271
3,naive_igg,naive_77_igg,10538
4,vap_ab,vap_38_ab,11050
5,vap_ab,vap_41_ab,10586
6,vap_ab,vap_49_ab,9673
7,vap_ab,vap_52_ab,10769
8,vap_igg,vap_42_igg,9173
9,vap_igg,vap_46_igg,10945


by_cell_type


,cluster_sub,n_cells
0,EC,67793
1,PERICYTE,18049
2,PVF,40172


by_compartment


,spatial_domain,condition,n_cells
0,0,naive_igg,3328
1,0,vap_ab,3661
2,0,vap_igg,3680
3,1,naive_igg,2853
4,1,vap_ab,2808
5,1,vap_igg,2979
6,10,naive_igg,2998
7,10,vap_ab,2800
8,10,vap_igg,3093
9,11,naive_igg,5428


## Output contract and commands

The notebook shows the expected files and generated cNMF commands before execution. The compact input stores raw counts in `X`, retains required metadata and coordinates, and records provenance hashes.

In [46]:
display(cnmf_status(spec, paths))
display(pd.Series({stage: " ".join(map(str, command)) for stage, command in cnmf_stage_commands(config, spec, paths).items()}, name="command").to_frame())

,stage,complete,observed,expected,artifact
0,input_export,True,5,5,/path/to/project-data/spatial-work...
1,sweep_prepare,True,2,2,/path/to/project-data/spatial-work...
2,sweep_factorize,True,315,315,/path/to/project-data/spatial-work...
3,sweep_combine,True,21,21,/path/to/project-data/spatial-work...
4,k_selection,True,2,2,/path/to/project-data/spatial-work...
5,selected_prepare,False,0,2,/path/to/project-data/spatial-work...
6,selected_factorize,False,0,100,/path/to/project-data/spatial-work...
7,selected_combine,False,0,1,/path/to/project-data/spatial-work...
8,consensus,False,0,6,/path/to/project-data/spatial-work...
9,usage_h5ad,False,0,1,/path/to/project-data/spatial-work...


,command
sweep_prepare,cnmf prepare --...
sweep_factorize_worker_0,cnmf factorize ...
sweep_combine,cnmf combine --...
sweep_kselect,cnmf k_selectio...
selected_prepare,cnmf prepare --...
selected_factorize_worker_0,cnmf factorize ...
selected_combine,cnmf combine --...
selected_consensus,cnmf consensus ...


## Export the cNMF input

Enable the export switch to write the selected raw-count matrix and its manifest. Existing complete outputs are reused unless overwrite is explicitly requested.

In [47]:
if RUN_EXPORT:
    manifest = export_lineage_counts(config, spec, paths, overwrite=OVERWRITE_EXPORT)
    display(pd.Series(manifest["output"], name="export"))
else:
    print("Set RUN_EXPORT=True to write the prepared counts object.")
display(cnmf_status(spec, paths))

Set RUN_EXPORT=True to write the prepared counts object.


,stage,complete,observed,expected,artifact
0,input_export,True,5,5,/path/to/project-data/spatial-work...
1,sweep_prepare,True,2,2,/path/to/project-data/spatial-work...
2,sweep_factorize,True,315,315,/path/to/project-data/spatial-work...
3,sweep_combine,True,21,21,/path/to/project-data/spatial-work...
4,k_selection,True,2,2,/path/to/project-data/spatial-work...
5,selected_prepare,False,0,2,/path/to/project-data/spatial-work...
6,selected_factorize,False,0,100,/path/to/project-data/spatial-work...
7,selected_combine,False,0,1,/path/to/project-data/spatial-work...
8,consensus,False,0,6,/path/to/project-data/spatial-work...
9,usage_h5ad,False,0,1,/path/to/project-data/spatial-work...


## Run the exploratory K sweep

The sweep runs cNMF preparation, factorization, combination, and the K-selection plot. Review the generated command first; use the tmux option for a resumable background run.

In [48]:
if RUN_SWEEP:
    if RUN_IN_TMUX:
        sweep_launcher = write_cnmf_tmux_scripts(CONFIG_PATH, LINEAGE, config, spec, paths, mode="sweep")
        display(pd.Series(sweep_launcher, name="sweep tmux launcher"))
        print("Paste into a terminal:")
        print(sweep_launcher["launch_command"])
        print("Then attach with:")
        print(sweep_launcher["attach_command"])
    else:
        if not paths.counts_h5ad.exists():
            export_lineage_counts(config, spec, paths)
        run_sweep(config, spec, paths)
else:
    print("Set RUN_SWEEP=True to run or resume the K sweep.")

if bool(cnmf_status(spec, paths).set_index("stage").loc["k_selection", "complete"]):
    k_stats = load_k_selection_stats(paths)
    display(k_stats)
    plot_k_selection(k_stats, selected_k=spec.selected_k).show()
else:
    print("K-selection outputs are not present yet.")

Set RUN_SWEEP=True to run or resume the K sweep.


,k,local_density_threshold,silhouette,prediction_error
0,10.0,0.5,0.855983,2.225411e+08
1,11.0,0.5,0.815162,2.219750e+08
2,12.0,0.5,0.912272,2.205516e+08
3,13.0,0.5,0.819957,2.198228e+08
4,14.0,0.5,0.865611,2.193170e+08
5,15.0,0.5,0.794099,2.184776e+08
6,16.0,0.5,0.794698,2.178526e+08
7,17.0,0.5,0.777004,2.173189e+08
8,18.0,0.5,0.839151,2.167443e+08
9,19.0,0.5,0.784242,2.160612e+08


## Fit the selected K

After reviewing the K-selection evidence, set `CHOSEN_K` and launch the higher-replicate consensus fit. This stage writes to a separate selected-K output directory.

In [49]:
CHOSEN_K = spec.selected_k
selected_spec, selected_paths = with_selected_k(CONFIG_PATH, config, spec, CHOSEN_K)

if RUN_SELECTED_CONSENSUS:
    if RUN_IN_TMUX:
        selected_launcher = write_cnmf_tmux_scripts(CONFIG_PATH, LINEAGE, config, selected_spec, selected_paths, mode="selected")
        display(pd.Series(selected_launcher, name="selected-K tmux launcher"))
        print("Paste into a terminal:")
        print(selected_launcher["launch_command"])
        print("Then attach with:")
        print(selected_launcher["attach_command"])
    else:
        if not selected_paths.counts_h5ad.exists():
            export_lineage_counts(config, selected_spec, selected_paths)
        run_selected(config, selected_spec, selected_paths)
else:
    print(
        "Set RUN_SELECTED_CONSENSUS=True to run or resume the "
        "higher-replicate selected-K fit."
    )
display(cnmf_status(selected_spec, selected_paths))

mode                                                        selected
job_script         /path/to/project-data/spatial-work...
launcher_script    /path/to/project-data/spatial-work...
log_path           /path/to/project-data/spatial-work...
session_name                              cnmf_perivascular_selected
launch_command     bash /path/to/project-data/spatial...
attach_command     tmux attach-session -t cnmf_perivascular_selected
Name: selected-K tmux launcher, dtype: object

Paste into a terminal:
bash /path/to/spatial-workflow/results/ab_xenium/05_cnmf/perivascular/launchers/launch_selected_tmux.sh
Then attach with:
tmux attach-session -t cnmf_perivascular_selected


,stage,complete,observed,expected,artifact
0,input_export,True,5,5,/path/to/project-data/spatial-work...
1,sweep_prepare,True,2,2,/path/to/project-data/spatial-work...
2,sweep_factorize,True,315,315,/path/to/project-data/spatial-work...
3,sweep_combine,True,21,21,/path/to/project-data/spatial-work...
4,k_selection,True,2,2,/path/to/project-data/spatial-work...
5,selected_prepare,False,0,2,/path/to/project-data/spatial-work...
6,selected_factorize,False,0,100,/path/to/project-data/spatial-work...
7,selected_combine,False,0,1,/path/to/project-data/spatial-work...
8,consensus,False,0,6,/path/to/project-data/spatial-work...
9,usage_h5ad,False,0,1,/path/to/project-data/spatial-work...


## Command-line alternative

The examples below run the same workflow without Jupyter. Repeat selection arguments as needed, and use `--lineage all` to process every configured lineage.

```bash
python3 scripts/run_cnmf_workflow.py \
  --config configs/local.yaml --lineage astrocyte --mode all
```